In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-03-25 13:20:27.040829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742908827.052735 2752188 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742908827.056328 2752188 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-25 13:20:27.070154: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1742908828.505048 2752188 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 17417 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-4:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-1e-4))

In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'lognumax':0.001, 'logdnuSer':0.001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [5]:
n_dense_layers = 2

dense_layer_units = 512

Nepochs = 5000

learning_rate = 0.01

model_name = 'smart-logLPhot-lognumax-logdnuSer-exponent-1e-4'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [6]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**16, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/5000


I0000 00:00:1742908857.987110 2753031 service.cc:148] XLA service 0x7336f50033a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742908857.987147 2753031 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-03-25 13:20:58.009599: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1742908858.070198 2753031 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-03-25 13:20:58.116889: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.8.61. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-03-25 13:20:58.67242

 13/104 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 354182.4375 

I0000 00:00:1742908860.441412 2753031 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 98/104 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 121457.0000

2025-03-25 13:21:01.940991: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 196 bytes spill stores, 196 bytes spill loads

2025-03-25 13:21:02.138622: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_239', 16 bytes spill stores, 16 bytes spill loads

2025-03-25 13:21:02.240587: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_239', 128 bytes spill stores, 128 bytes spill loads

2025-03-25 13:21:02.327476: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122_0', 756 bytes spill stores, 444 bytes spill loads

2025-03-25 13:21:02.394846: I external/local_xla/xla/str

104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 117236.7422

2025-03-25 13:21:04.484648: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 196 bytes spill stores, 196 bytes spill loads

2025-03-25 13:21:04.712008: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 740 bytes spill stores, 436 bytes spill loads




Epoch 1: val_loss improved from inf to 19746.63477, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-1e-4-nlayers-2-nunits-512-epochs-5000-lrate-0.01-lossfunc-WMSE.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - loss: 116575.6875 - val_loss: 19746.6348 - learning_rate: 0.0100
Epoch 2/5000
103/104 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 19560.4121
Epoch 2: val_loss improved from 19746.63477 to 19143.60547, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-1e-4-nlayers-2-nunits-512-epochs-5000-lrate-0.01-lossfunc-WMSE.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 19557.1934 - val_loss: 19143.6055 - learning_rate: 0.0100
Epoch 3/5000
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 19138.5371
Epoch 3: val_loss improved from 19143.60547 to 19042.55469, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-expon